# 무연고 사망, 독거노인 이야기일까? 서울 4개 구 데이터로 본 50~60대 남성

DataPopcorn 테크블로그 글의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 표와 차트 6개가 다시 만들어집니다.

- 무연고 사망자: 공공데이터포털 서울 중랑(15133632)·동대문(15133646)·성북(15132709)·용산(15132710)·은평(15132694)·광진(15133976)구 파일 원본
- 사망자 수: 국가데이터처 인구동향조사(사망), 한국여성정책연구원 성인지통계 DT_2FB0211R에서 내려받아 정리
- 인구: KOSIS 주민등록인구 DT_1B04005N(연말)
- 조회일: 2026-11-03. 로그인·API 키 없이 공개 저장소에서 읽습니다.

In [ ]:
# 0. 준비: 한글 폰트와 데이터 위치
import os, sys, io, re, csv, urllib.request
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if 'google.colab' in sys.modules:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        fm.fontManager.addfont(f); plt.rcParams['font.family'] = fm.FontProperties(fname=f).get_name(); break
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
C = {'main': '#E8590C', 'sub': '#1C7ED6', 'gray': '#868E96'}
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/seoul-unclaimed-deaths/data')
def read_bytes(rel):
    p = f'{BASE}/{rel}'
    return urllib.request.urlopen(p).read() if p.startswith('http') else open(p, 'rb').read()
def read_text(rel):
    b = read_bytes(rel)
    for enc in ('utf-8-sig', 'cp949'):
        try: return b.decode(enc).lstrip('\ufeff')
        except UnicodeDecodeError: pass
def read_rows(rel):
    rows = list(csv.reader(io.StringIO(read_text(rel))))
    hdr = [re.sub(r'^.*?(시도명)$', r'\1', h.strip()) for h in rows[0]]   # 성북구 파일의 깨진 BOM 정리
    return [dict(zip(hdr, r)) for r in rows[1:] if any(r)]
print('데이터 경로:', BASE)

## Step 3~4. 여섯 구의 파일을 하나의 긴 표로 합칩니다

파일마다 모양이 다릅니다. 모두 `구, 연도, 성별, 연령대, 인원`으로 바꾸고 원표 합계와 대조합니다.

In [ ]:
def age10(label):
    label = label.replace(' ', '')
    if label in ('미상', '기타', '미상및기타'): return '미상'
    m = re.match(r'(\d+)', label)
    if not m: return None
    a = int(m.group(1)) // 10 * 10
    return '0-29' if a < 30 else ('90+' if a >= 90 else f'{a}-{a+9}')
out = []
def add(gu, year, sex, age, n):
    if n not in ('', None): out.append(dict(gu=gu, year=str(year), sex=sex, age10=age, n=int(float(n))))
for r in read_rows('raw/gu_15133632.csv'):        # 중랑구
    if r['성별'] != '계' and r['연령별'] != '계':
        for y in range(2017, 2024): add('중랑구', y, r['성별'], age10(r['연령별']), r[str(y)])
for r in read_rows('raw/gu_15132709.csv'):        # 성북구
    for k, v in r.items():
        m = re.match(r'(\d+)대\((남|여)\)', k or '')
        if m: add('성북구', r['연도'], m.group(2), age10(m.group(1) + '대'), v)
        elif k and k.startswith('미상'): add('성북구', r['연도'], '미상', '미상', v)
for r in read_rows('raw/gu_15133646.csv'):        # 동대문구 (0명 행은 파일에 없음)
    add('동대문구', r['연도'], '남', age10(r['연령대']), r['남자 사망자수']); add('동대문구', r['연도'], '여', age10(r['연령대']), r['여자 사망자수'])
for r in read_rows('raw/gu_15132694.csv'):        # 은평구 (성별 없음)
    add('은평구', r['연도'], '미상', age10(r['나이대']), r['사망자수'])
for r in read_rows('raw/gu_15132710.csv'):        # 용산구
    if r['구분'] == '전체': continue
    for k, v in r.items():
        m = re.match(r'(\d{4})년', k or '')
        if not m: continue
        if r['구분'] == '미상': add('용산구', m.group(1), '미상', '미상', v)
        else:
            mm = re.match(r'(\d+)대\((남|여)\)', r['구분']); add('용산구', m.group(1), mm.group(2), age10(mm.group(1) + '대'), v)
for r in read_rows('raw/gu_15133976.csv'):        # 광진구 (2020.1~2025.6 누적)
    lab = r['나이대'].replace(' ', '')
    a = '70+' if lab.startswith('70세') else ('60-69' if lab[:3] in ('60세', '65세') else age10(lab))
    add('광진구', '2020-01~2025-06', '남', a, r['남성']); add('광진구', '2020-01~2025-06', '여', a, r['여성'])
u = pd.DataFrame(out)
print('행 수:', len(u))
print(u.groupby(['gu', 'year']).n.sum().unstack(0).fillna(0).astype(int).to_string())
assert u[u.gu == '중랑구'].groupby('year').n.sum().tolist() == [50, 67, 37, 49, 52, 52, 79]
assert u[u.gu == '용산구'].groupby('year').n.sum().tolist() == [28, 38, 50, 52, 59, 66, 83, 69]
assert u[u.gu == '광진구'].n.sum() == 209

## Step 5~6. 누가 무연고 사망자가 되었나, 분모를 붙이면 어떻게 보이나

성별·연령·연도가 모두 있는 4개 구(중랑·동대문·성북·용산)만 씁니다. 같은 구·연도·성·연령의 사망자 수와 인구를 붙입니다.

In [ ]:
d = pd.read_csv(io.BytesIO(read_bytes('denominators/deaths_long.csv')))
p = pd.read_csv(io.BytesIO(read_bytes('denominators/population_long.csv')))
p['age10'] = p['age10'].replace({'100+': '90+'}); p = p.groupby(['gu', 'year', 'sex', 'age10'], as_index=False)['pop'].sum()
GRP = {'0-9':'0-29','10-19':'0-29','20-29':'0-29','0-29':'0-29','30-39':'30-39','40-49':'40-49','50-59':'50-59','60-69':'60-69','70-79':'70-79','80-89':'80-89','90+':'90+'}
SCOPE = {'중랑구': range(2017, 2024), '동대문구': range(2017, 2024), '성북구': range(2017, 2026), '용산구': range(2018, 2026)}
def scope(df, scope=SCOPE):
    return pd.concat([df[(df.gu == g) & df.year.astype(str).isin([str(y) for y in ys])] for g, ys in scope.items()])
u2 = scope(u[u.sex.isin(['남', '여']) & (u.age10 != '미상')]).copy(); u2['ag'] = u2.age10.map(GRP)
d2 = scope(d).copy(); d2['ag'] = d2.age10.map(GRP); p2 = scope(p).copy(); p2['ag'] = p2.age10.map(GRP)
t = pd.concat([u2.groupby(['sex','ag']).n.sum().rename('unclaimed'), d2.groupby(['sex','ag']).deaths.sum().rename('deaths'), p2.groupby(['sex','ag'])['pop'].sum().rename('person_years')], axis=1).fillna(0)
t['per_1000_deaths'] = t.unclaimed / t.deaths * 1000
t['per_100k_pop'] = t.unclaimed / t.person_years * 1e5
t = t.reset_index()
print(t.round(1).to_string())
m = t[(t.sex == '남') & t.ag.isin(['50-59', '60-69'])]
print(f"\n무연고 {int(t.unclaimed.sum()):,}명 중 50~60대 남성 {int(m.unclaimed.sum())}명 ({m.unclaimed.sum()/t.unclaimed.sum():.1%})")
print(f"50~60대 남성의 인구 비중 {m.person_years.sum()/t.person_years.sum():.1%}, 사망자 비중 {m.deaths.sum()/t.deaths.sum():.1%}")
print(f"50~60대 남성 사망자 1,000명당 무연고 {m.unclaimed.sum()/m.deaths.sum()*1000:.1f}명, 전체 {t.unclaimed.sum()/t.deaths.sum()*1000:.1f}명")

In [ ]:
ages = ['0-29','30-39','40-49','50-59','60-69','70-79','80-89','90+']; lab = ['29세 이하','30대','40대','50대','60대','70대','80대','90세 이상']
M = t[t.sex == '남'].set_index('ag').reindex(ages); F = t[t.sex == '여'].set_index('ag').reindex(ages)
x = np.arange(len(ages)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].bar(x - w/2, M.unclaimed, w, color=[C['main'] if a in ('50-59','60-69') else '#F4A47A' for a in ages], label='남성')
ax[0].bar(x + w/2, F.unclaimed, w, color=C['sub'], alpha=.75, label='여성'); ax[0].set_xticks(x, lab, fontsize=8); ax[0].legend(frameon=False)
ax[0].set_title('① 무연고 사망자 수: 50~60대 남성이 절반', loc='left')
vals = [m.person_years.sum()/t.person_years.sum(), m.deaths.sum()/t.deaths.sum(), m.unclaimed.sum()/t.unclaimed.sum()]
ax[1].barh(['인구 중', '사망자 중', '무연고 사망자 중'][::-1], [v*100 for v in vals[::-1]], color=[C['main'], C['gray'], C['gray']])
for i, v in enumerate(vals[::-1]): ax[1].text(v*100 + .8, i, f'{v:.1%}', va='center')
ax[1].set_title('② 50~60대 남성이 차지하는 비중', loc='left'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].plot(lab, M.per_1000_deaths, color=C['main'], marker='o', lw=2.4, label='남성'); ax[0].plot(lab, F.per_1000_deaths, color=C['sub'], marker='o', label='여성')
ax[0].set_title('③ 사망자 1,000명당 무연고: 50~60대 남성 최고', loc='left'); ax[0].legend(frameon=False); ax[0].tick_params(axis='x', labelsize=8)
ax[1].bar(x - w/2, M.per_100k_pop, w, color='#F4A47A', label='남성'); ax[1].bar(x + w/2, F.per_100k_pop, w, color=C['sub'], alpha=.75, label='여성')
ax[1].set_xticks(x, lab, fontsize=8); ax[1].legend(frameon=False); ax[1].set_title('④ 인구 10만 명당: 분모가 바뀌면 고령 남성이 더 높다', loc='left')
plt.tight_layout(); plt.show()

## Step 8. 구마다 같은 모양인지, 기록이 끊기지 않았는지 봅니다

In [ ]:
G = {'0-9':'0-29','10-19':'0-29','20-29':'0-29','0-29':'0-29','30-39':'30-49','40-49':'30-49','50-59':'50-69','60-69':'50-69','70-79':'70+','80-89':'70+','90+':'70+','70+':'70+'}
def run(sc, label):
    uu = scope(u[u.sex.isin(['남','여']) & (u.age10 != '미상')], sc).copy(); uu['g'] = uu.age10.map(G)
    dd = scope(d, sc).copy(); dd['g'] = dd.age10.map(G)
    U = uu.groupby(['sex','g']).n.sum(); D = dd.groupby(['sex','g']).deaths.sum()
    r = lambda s, a: U.get((s, a), 0) / D.get((s, a), 1) * 1000
    return dict(조건=label, 무연고=int(U.sum()), 남50_69_비중=round(U.get(('남','50-69'),0)/U.sum()*100, 1), 남50_69_사망천명당=round(r('남','50-69'), 1), 남70이상_사망천명당=round(r('남','70+'), 1))
rows = [run(SCOPE, '기본(4개 구)'), run({k: v for k, v in SCOPE.items() if k != '성북구'}, '성북구 제외'),
        run({**SCOPE, '성북구': range(2023, 2026)}, '성북구 2023~2025만'), run({k: v for k, v in SCOPE.items() if k != '용산구'}, '용산구 제외'),
        run({g: [y for y in ys if y <= 2020] for g, ys in SCOPE.items()}, '2017~2020'), run({g: [y for y in ys if y >= 2021] for g, ys in SCOPE.items()}, '2021~2025')]
rows += [run({g: ys}, f'{g} 단독') for g, ys in SCOPE.items()]
sens = pd.DataFrame(rows); print(sens.to_string(index=False))
yr = u[u.gu.isin(SCOPE) & u.year.str.isdigit()].assign(year=lambda df: df.year.astype(int)).groupby(['year','gu']).n.sum().unstack()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
g4 = sens[sens.조건.str.contains('단독')]
ax[0].barh(g4.조건, g4.남50_69_사망천명당, color=C['main'], label='남성 50~69세'); ax[0].barh(g4.조건, g4.남70이상_사망천명당, color=C['gray'], height=.4, label='남성 70세 이상')
ax[0].legend(frameon=False); ax[0].set_title('⑤ 구별 사망자 1,000명당 무연고', loc='left')
for g in yr.columns: ax[1].plot(yr.index, yr[g], marker='o', lw=2.4 if g == '성북구' else 1.2, label=g, color=C['main'] if g == '성북구' else None)
ax[1].legend(frameon=False, ncol=4, fontsize=8); ax[1].set_title('⑥ 연간 무연고 사망자 수: 성북구 2023년 계단', loc='left')
plt.tight_layout(); plt.show()

## 남은 질문

- 이 데이터에는 원인(실직, 이혼, 질병, 주거)이 없습니다. '누가'까지만 답합니다.
- 서울 25개 구 중 6개 구만 공개했습니다. 다른 구가 같은 형식으로 공개하면 `SCOPE`에 추가해 다시 돌리면 됩니다.
- 분모 원자료의 조회 방법은 `data/denominators/retrieval.md`에 있습니다.